# Federated Generative Learning

In [1]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
import torch

sys.path.insert(0, '..')
from UC1Utils import ensure_data

from UC1FLUtils import (
    MLP, Generator,
    load_clients,
    save_result,
    run_fedgen_full,
    run_fedgen_partial,
    evaluate_global,
)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Paths ──────────────────────────────────────────────────────────────────────
FEDERATED_DIR  = '../federated_data'
FILTERED_DIR   = os.path.join(FEDERATED_DIR, 'filtered')
UNFILTERED_DIR = os.path.join(FEDERATED_DIR, 'unfiltered')
RESULTS_DIR    = 'results'

CENTRALIZED_AUC = 0.658

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs('figures',   exist_ok=True)

# ── Shared hyperparameters — loaded from FedAvg tuning ────────────────────────
_hp_path = os.path.join(FEDERATED_DIR, 'fl_hyperparams.json')
if not os.path.exists(_hp_path):
    raise FileNotFoundError(
        f'Hyperparameters not found at {_hp_path}.\n'
        f'Run 02_UC1_Federated.ipynb first.'
    )
with open(_hp_path) as f:
    _hp = json.load(f)

HIDDEN_DIM = _hp['hidden_dim']
DROPOUT    = _hp['dropout']
LR         = _hp['lr']
BATCH_SIZE = _hp['batch_size']

print(f'Hyperparameters: hidden_dim={HIDDEN_DIM} dropout={DROPOUT:.4f} '
      f'lr={LR:.6f} batch_size={BATCH_SIZE}')

# ── FedGen-specific hyperparameters ───────────────────────────────────────────
N_CLIENTS    = 5
FL_ROUNDS    = 30
LOCAL_EPOCHS = 10
PATIENCE     = 5
NOISE_DIM    = 16
GEN_LR       = 1e-3
GEN_STEPS    = 10
LAMBDA_PROTO = 0.1
SEEDS        = [42, 123, 7]
ALPHA_SWEEP  = [0.1, 0.5, 1.0, 5.0, 10.0]

latent_dim = HIDDEN_DIM // 4
print(f'Latent dim: {latent_dim}  (HIDDEN_DIM={HIDDEN_DIM} // 4)')

/mnt/c/Users/Marc/Documents/Uni/4t/TFG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
Hyperparameters: hidden_dim=512 dropout=0.1803 lr=0.009296 batch_size=256
Latent dim: 128  (HIDDEN_DIM=512 // 4)


## KD-gen: Knowledge Distillation with Generative Model

Adapted from Salami et al. (arXiv:2405.05140v1) for UC1 binary classification.

### Key idea
Standard FedAvg suffers from **client drift** under non-IID data (α=0.5 Dirichlet partitioning).
KD-gen addresses this by introducing a lightweight generator G_ω shared across clients that
produces synthetic latent vectors Ẑ, used as an inductive bias during local training.

### What changes vs FedAvg

| | FedAvg | KD-gen |
|---|---|---|
| Communicated per round | Full model θ | Predictor **g_r** + generator **ω** |
| Feature extractor g_f | Aggregated on server | **Stays local** per client |
| Synthetic data | None | G_ω(y, ε) → Ẑ used in KD loss |
| Rationale for small G_ω | — | Tabular binary task; g_f already maps 99 features to ℝ¹²⁸ |

### Algorithm 1 (adapted for classification)
```
for round t:
  server  → clients : g_r weights, generator ω
  each client k:
    load global g_r  (g_f stays local)
    sample Ŷ ~ U{0,1},  ε ~ N(0,I)
    Ẑ = G_ω(Ŷ, ε)           ← frozen, no grad
    for each local epoch:
      step 1 – full model update:  L_real = CE(g_r(g_f(X)), Y)
      step 2 – g_r only update :   L_kd   = CE(g_r(Ẑ), Ŷ)
    send g_r state + per-class latent distribution p^(k) to server
  server:
    FedAvg on g_r  (weighted by client sample sizes)
    GMM update:  p̂ ← weighted mean of {p^(k)}
    generator update:  min CE(ensemble(G_ω(Ŷ',ε')), Ŷ') + λ·||G_ω(Ŷ',ε') - z̄_{Ŷ'}||²
```

In [2]:
# ── Parameter counts and communication cost ────────────────────────────────────
# Instantiate probe models to count parameters — immediately deleted after.
_m = MLP(99, HIDDEN_DIM, dropout=DROPOUT)   # 99 = placeholder input_dim
_g = Generator(latent_dim, NOISE_DIM)

n_full       = sum(p.numel() for p in _m.parameters())
n_predictor  = sum(p.numel() for p in _m.predictor.parameters())
n_gen        = sum(p.numel() for p in _g.parameters())
del _m, _g

B = 4  # float32 bytes

bytes_fedavg_full    = 2 * n_full                      * B * N_CLIENTS
bytes_fedgen_full    = (2 * n_full        + n_gen)     * B * N_CLIENTS
bytes_fedgen_partial = (2 * n_predictor   + n_gen)     * B * N_CLIENTS

print(f'{"Component":<25} {"Params":>10} {"KB":>8}')
print('-' * 46)
print(f'{"Full MLP":<25} {n_full:>10,} {n_full*B/1024:>8.1f}')
print(f'{"  Feature extractor":<25} {n_full-n_predictor:>10,} {(n_full-n_predictor)*B/1024:>8.1f}')
print(f'{"  Predictor head":<25} {n_predictor:>10,} {n_predictor*B/1024:>8.1f}')
print(f'{"Generator":<25} {n_gen:>10,} {n_gen*B/1024:>8.1f}')
print(f'\n{"Variant":<25} {"MB/round":>10} {"vs FedAvg full":>16}')
print('-' * 54)
print(f'{"FedAvg full":<25} {bytes_fedavg_full/1024**2:>10.2f} {"(baseline)":>16}')
print(f'{"FedGen full":<25} {bytes_fedgen_full/1024**2:>10.2f} '
      f'{bytes_fedgen_full/bytes_fedavg_full:>15.2f}×')
print(f'{"FedGen partial":<25} {bytes_fedgen_partial/1024**2:>10.2f} '
      f'{bytes_fedgen_partial/bytes_fedavg_full:>14.3f}×')
print(f'\n→ FedGen partial is {bytes_fedavg_full/bytes_fedgen_partial:.0f}× cheaper than FedAvg full.')

Component                     Params       KB
----------------------------------------------
Full MLP                     217,474    849.5
  Feature extractor          217,216    848.5
  Predictor head                 258      1.0
Generator                      4,904     19.2

Variant                     MB/round   vs FedAvg full
------------------------------------------------------
FedAvg full                     8.30       (baseline)
FedGen full                     8.39            1.01×
FedGen partial                  0.10          0.012×

→ FedGen partial is 80× cheaper than FedAvg full.


In [ ]:
re_train = False

for data_case, data_dir in [('filtered', FILTERED_DIR), ('unfiltered', UNFILTERED_DIR)]:
    print(f'\n{"#"*60}')
    print(f'Data case: {data_case}')
    print(f'{"#"*60}')

    for alpha in ALPHA_SWEEP:
        for seed in SEEDS:
            # r_base is what save_result receives — it appends alpha/seed internally
            r_base       = os.path.join(RESULTS_DIR, data_case)
            # r_dir is the full path used only for existence checks
            r_dir        = os.path.join(r_base, f'alpha_{alpha}', f'seed_{seed}')
            partial_path = os.path.join(r_dir, 'fedgen_partial.json')
            full_path    = os.path.join(r_dir, 'fedgen_full.json')

            if os.path.exists(partial_path) and os.path.exists(full_path) and not re_train:
                print(f'  [{data_case}] α={alpha} seed={seed}: already done, skipping.')
                continue

            print(f'\n{"="*60}')
            print(f'FedGen  [{data_case}]  α={alpha}  seed={seed}')
            print(f'{"="*60}')

            try:
                clients = load_clients(alpha, data_dir, N_CLIENTS)
            except (FileNotFoundError, ValueError) as e:
                print(f'  Skipping: {e}')
                continue

            input_dim = clients[0]['X_train'].shape[1]

            if not os.path.exists(partial_path) or re_train:
                print('\n--- fedgen_partial ---')
                auc, pc, hist, cmb = run_fedgen_partial(
                    clients, input_dim, seed,
                    hidden_dim=HIDDEN_DIM, dropout=DROPOUT,
                    lr=LR, batch_size=BATCH_SIZE,
                    n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
                    local_epochs=LOCAL_EPOCHS, patience=PATIENCE,
                    noise_dim=NOISE_DIM,
                )
                save_result(r_base, alpha, seed, 'fedgen_partial', auc, pc, hist, cmb)

            if not os.path.exists(full_path) or re_train:
                print('\n--- fedgen_full ---')
                auc, pc, hist, cmb = run_fedgen_full(
                    clients, input_dim, seed,
                    hidden_dim=HIDDEN_DIM, dropout=DROPOUT,
                    lr=LR, batch_size=BATCH_SIZE,
                    n_clients=N_CLIENTS, fl_rounds=FL_ROUNDS,
                    local_epochs=LOCAL_EPOCHS, patience=PATIENCE,
                    noise_dim=NOISE_DIM,
                )
                save_result(r_base, alpha, seed, 'fedgen_full', auc, pc, hist, cmb)

print('\nAll FedGen experiments complete.')


############################################################
Data case: filtered
############################################################

FedGen  [filtered]  α=0.1  seed=42
  client_0: 28,959 train  pos_rate=6.8%  n_pos=1977
  client_1: 2,472 train  pos_rate=43.7%  n_pos=1080
  client_2: 22,830 train  pos_rate=5.0%  n_pos=1141
  client_3: 6,799 train  pos_rate=43.2%  n_pos=2939
  client_4: 5,401 train  pos_rate=9.0%  n_pos=488

--- fedgen_partial ---
  [fedgen_partial] R01: val=0.4475 test=0.4415 cumul=0.10MB
  [fedgen_partial] R02: val=0.5960 test=0.5833 cumul=0.21MB
  [fedgen_partial] R03: val=0.6282 test=0.6231 cumul=0.31MB
  [fedgen_partial] R04: val=0.6486 test=0.6377 cumul=0.41MB
  [fedgen_partial] R05: val=0.6655 test=0.6568 cumul=0.52MB
  [fedgen_partial] R06: val=0.6481 test=0.6387 cumul=0.62MB
  [fedgen_partial] R07: val=0.6816 test=0.6710 cumul=0.72MB
  [fedgen_partial] R08: val=0.6673 test=0.6659 cumul=0.83MB
  [fedgen_partial] R09: val=0.6507 test=0.6448 cumul=0.93MB

In [5]:
import glob

print(f'{"Case":<12} {"Alpha":>6}  {"Seed":>6}  {"Variant":<20}  '
      f'{"Test AUC":>9}  {"Rounds":>7}')
print('-' * 72)

for path in sorted(glob.glob(f'{RESULTS_DIR}/*/*/seed_*/*.json')):
    r      = json.load(open(path))
    # path: results/filtered/alpha_0.5/seed_42/fedgen_partial.json
    parts  = path.replace(RESULTS_DIR + '/', '').replace('.json', '').split('/')
    case, alpha_s, seed_s, variant = parts
    print(f'{case:<12} {alpha_s.replace("alpha_",""):>6}  '
          f'{seed_s.replace("seed_",""):>6}  '
          f'{variant:<20}  '
          f'{r["test_auc"]:>9.4f}  '
          f'{len(r["history"]):>7}')

Case          Alpha    Seed  Variant                Test AUC   Rounds
------------------------------------------------------------------------
